In [ ]:
import pandas as pd
import numpy as np
from scipy.linalg import eigvalsh
import json

# Load withdrawal matrix
matrix = pd.read_csv("data/withdrawal_matrix_1h.csv", index_col=0, parse_dates=True)
matrix.index = pd.to_datetime(matrix.index, utc=True)
matrix = matrix.astype(float)  # Ensure all values are numeric

print(f"Matrix shape: {matrix.shape}")
print(f"First 3 rows, first 5 columns:\n{matrix.iloc[:3, :5]}")


# Test on a single window
test_window = matrix.iloc[0:24]
T, N = test_window.shape

print(f"\nT={T}, N={N}, T >= N: {T >= N}")
print(f"Total withdrawals: {test_window.sum().sum():.2f}")

# Standardization
std = test_window.std()
std[std < 1e-10] = 1e-10
normed = (test_window - test_window.mean()) / std

# Compute correlation matrix
corr = (normed.T @ normed / T).values

print(f"Corr matrix shape: {corr.shape}")
print(f"Number of NaN in corr: {np.isnan(corr).sum()}")
print(f"Number of Inf in corr: {np.isinf(corr).sum()}")

# Eigenvalues
evals = np.sort(eigvalsh(corr))[::-1]
print(f"Top 5 eigenvalues: {evals[:5].round(4)}")

# Marchenko-Pastur upper bound
gamma = N / T
lambda_plus = (1 + np.sqrt(gamma)) ** 2

print(f"gamma = {gamma:.3f}, lambda_plus = {lambda_plus:.4f}")

In [ ]:
import pandas as pd
import numpy as np
from scipy.linalg import eigvalsh
import json

matrix = pd.read_csv("data/withdrawal_matrix_1h.csv", index_col=0, parse_dates=True)
matrix.index = pd.to_datetime(matrix.index, utc=True)
matrix = matrix.astype(float)

def compute_rmt(window_df):
    T, N = window_df.shape
    if window_df.sum().sum() == 0:
        return None, None, None, None
    std = window_df.std()
    std[std < 1e-10] = 1e-10
    normed = (window_df - window_df.mean()) / std
    corr = (normed.T @ normed / T).values
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
    evals = np.sort(eigvalsh(corr))[::-1]
    gamma = N / T
    lambda_plus = (1 + np.sqrt(gamma)) ** 2
    eff_rank = np.exp(-np.sum((evals/evals.sum()) * np.log(evals/evals.sum() + 1e-10)))
    mean_corr = (corr.sum() - N) / (N * (N - 1))
    return float(evals[0]), float(lambda_plus), float(eff_rank), float(mean_corr)

WINDOW = 48  # T=48 > N=29，矩阵满秩
results = []

for i in range(WINDOW, len(matrix)):
    lmax, lplus, er, mc = compute_rmt(matrix.iloc[i-WINDOW:i])
    if lmax is not None:
        results.append({
            "timestamp":   matrix.index[i].isoformat(),
            "lambda_max":  lmax,
            "lambda_plus": lplus,
            "eff_rank":    er,
            "mean_corr":   mc,
        })

df_rmt = pd.DataFrame(results)
print(f"RMT序列长度: {len(df_rmt)}")

# ── 关键窗口 ──────────────────────────────────────────
df_rmt["ts"] = pd.to_datetime(df_rmt["timestamp"], utc=True)
t_start = pd.Timestamp("2026-04-18 15:00:00", tz="UTC")
t_end   = pd.Timestamp("2026-04-18 21:00:00", tz="UTC")
window  = df_rmt[(df_rmt["ts"] >= t_start) & (df_rmt["ts"] <= t_end)]

print("\n=== 关键窗口 15:00-21:00 Apr 18 ===")
print(window[["timestamp","lambda_max","lambda_plus","eff_rank","mean_corr"]].to_string(index=False))

print("\n=== 全序列λ₁最大5个时刻 ===")
print(df_rmt.nlargest(5,"lambda_max")[["timestamp","lambda_max","lambda_plus","eff_rank"]].to_string(index=False))

# ── 导出JSON ─────────────────────────────────────────
json_out = {
    "attack_time": "2026-04-18T17:35:35+00:00",
    "news_time":   "2026-04-18T19:00:00+00:00",
    "timestamps":  df_rmt["timestamp"].tolist(),
    "lambda_max":  df_rmt["lambda_max"].tolist(),
    "lambda_plus": df_rmt["lambda_plus"].tolist(),
    "eff_rank":    df_rmt["eff_rank"].tolist(),
    "mean_corr":   df_rmt["mean_corr"].tolist(),
}
with open("data/rmt_dashboard.json", "w") as f:
    json.dump(json_out, f)
print("\n✓ 保存至 data/rmt_dashboard.json")

In [ ]:
import pandas as pd
import numpy as np
from scipy.linalg import eigvalsh
import json

# Load withdrawal matrix
matrix = pd.read_csv("data/withdrawal_matrix_1h.csv", index_col=0, parse_dates=True)
matrix.index = pd.to_datetime(matrix.index, utc=True)
matrix = matrix.astype(float)

def compute_rmt(window_df):
    """
    Compute RMT metrics for a given time window
    """
    T, N = window_df.shape
    if window_df.sum().sum() == 0:
        return None, None, None, None
    
    std = window_df.std()
    std[std < 1e-10] = 1e-10
    normed = (window_df - window_df.mean()) / std
    
    corr = (normed.T @ normed / T).values
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
    
    evals = np.sort(eigvalsh(corr))[::-1]
    gamma = N / T
    lambda_plus = (1 + np.sqrt(gamma)) ** 2
    
    # Effective Rank
    p = evals / evals.sum()
    eff_rank = np.exp(-np.sum(p * np.log(p + 1e-10)))
    
    # Mean correlation (off-diagonal)
    mean_corr = (corr.sum() - N) / (N * (N - 1))
    
    return float(evals[0]), float(lambda_plus), float(eff_rank), float(mean_corr)


WINDOW = 48  # T=48 > N=29, ensures full rank

results = []
for i in range(WINDOW, len(matrix)):
    lmax, lplus, er, mc = compute_rmt(matrix.iloc[i-WINDOW:i])
    if lmax is not None:
        results.append({
            "timestamp": matrix.index[i].isoformat(),
            "lambda_max": lmax,
            "lambda_plus": lplus,
            "eff_rank": er,
            "mean_corr": mc,
        })

df_rmt = pd.DataFrame(results)
print(f"RMT sequence length: {len(df_rmt)}")


# ── Key Time Window Analysis ─────────────────────────────────────
df_rmt["ts"] = pd.to_datetime(df_rmt["timestamp"], utc=True)

t_start = pd.Timestamp("2026-04-18 15:00:00", tz="UTC")
t_end = pd.Timestamp("2026-04-18 21:00:00", tz="UTC")

window = df_rmt[(df_rmt["ts"] >= t_start) & (df_rmt["ts"] <= t_end)]

print("\n=== Key Window 15:00–21:00 on Apr 18 ===")
print(window[["timestamp", "lambda_max", "lambda_plus", "eff_rank", "mean_corr"]].to_string(index=False))


# ── Top 5 Moments with Highest λ₁ ───────────────────────────────
print("\n=== Top 5 Moments with Highest λ₁ ===")
print(df_rmt.nlargest(5, "lambda_max")[["timestamp", "lambda_max", "lambda_plus", "eff_rank"]].to_string(index=False))


# ── Export to JSON for Dashboard ───────────────────────────────
json_out = {
    "attack_time": "2026-04-18T17:35:35+00:00",
    "news_time": "2026-04-18T19:00:00+00:00",
    "timestamps": df_rmt["timestamp"].tolist(),
    "lambda_max": df_rmt["lambda_max"].tolist(),
    "lambda_plus": df_rmt["lambda_plus"].tolist(),
    "eff_rank": df_rmt["eff_rank"].tolist(),
    "mean_corr": df_rmt["mean_corr"].tolist(),
}

with open("data/rmt_dashboard.json", "w") as f:
    json.dump(json_out, f, indent=2)

print("\n✓ Saved to data/rmt_dashboard.json")

In [ ]:
with open("data/rmt_dashboard.json") as f:
    content = f.read()
print(content)

In [ ]:
# ==================== Fix Chinese Character Display Issues ====================
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# Set fonts that support Chinese characters (try common system fonts in order)
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'Microsoft YaHei',
                                   'PingFang SC', 'Hiragino Sans GB', 'DejaVu Sans']

# Fix negative sign display issue
plt.rcParams['axes.unicode_minus'] = False

# ======================================================

In [ ]:
# ══════════════════════════════════════════════════════
# Step 1: Effective Rank Rolling Time Series (Panic Index)
# ══════════════════════════════════════════════════════

rolling_results = []
WINDOW = 12  # 12-hour rolling window (T=12 < N=29, correlation still computable)

for i in range(WINDOW, len(matrix)):
    w = matrix.iloc[i-WINDOW:i]
    if w.sum().sum() == 0:
        continue
    r = get_spectrum(w)
    rolling_results.append({
        "timestamp": matrix.index[i],
        "lambda_max": r["evals"][0],
        "lambda_plus": r["lplus"],
        "eff_rank": r["eff_rank"],
        "mean_corr": r["mean_corr"],
        "n_above_mp": r["n_above_mp"],
    })

df_roll = pd.DataFrame(rolling_results)
df_roll.to_csv("output/rolling_rmt.csv", index=False)

# ── Figure 2: Quad Time Series Plot ─────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)
fig.subplots_adjust(hspace=0.25)

t = df_roll["timestamp"]
ATTACK_T = pd.Timestamp("2026-04-18T17:35:35", tz="UTC")
NEWS_T = pd.Timestamp("2026-04-18T19:00:00", tz="UTC")

def add_events(ax):
    ax.axvline(ATTACK_T, color="#FF9800", lw=1.5, ls="--", alpha=0.8, label="17:35 Attack")
    ax.axvline(NEWS_T, color="#FFD700", lw=1.5, ls=":", alpha=0.8, label="19:00 News")

# λ₁ - Largest Eigenvalue
axes[0].plot(t, df_roll["lambda_max"], color="#00aaff", lw=1.5)
axes[0].fill_between(t, df_roll["lambda_max"], alpha=0.15, color="#00aaff")
axes[0].axhline(LP, color="royalblue", ls="--", lw=1.5, label=f"λ+={LP:.2f}")
axes[0].fill_between(t, df_roll["lambda_max"],
                     where=df_roll["lambda_max"] > LP, color="#ff2255", alpha=0.25, label="Above MP")
add_events(axes[0])
axes[0].set_ylabel("λ₁")
axes[0].legend(fontsize=8, loc="upper left")
axes[0].set_title("λ₁ Largest Eigenvalue (Exceeding M-P Bound = Systemic Signal)", fontsize=11)

# Effective Rank
axes[1].plot(t, df_roll["eff_rank"], color="#00dd88", lw=1.5)
axes[1].fill_between(t, df_roll["eff_rank"], alpha=0.15, color="#00dd88")
axes[1].axhline(8, color="orange", ls="--", lw=1, label="Panic Threshold = 8")
add_events(axes[1])
axes[1].set_ylabel("Effective Rank")
axes[1].legend(fontsize=8, loc="upper right")
axes[1].set_title("Effective Rank (Decline = Homogenization of Withdrawals = Panic Contagion)", fontsize=11)

# Mean Correlation
axes[2].plot(t, df_roll["mean_corr"], color="#ff9933", lw=1.5)
axes[2].fill_between(t, df_roll["mean_corr"], alpha=0.15, color="#ff9933")
axes[2].axhline(0.25, color="red", ls="--", lw=1, label="Panic Threshold = 0.25")
add_events(axes[2])
axes[2].set_ylabel("Mean Correlation")
axes[2].legend(fontsize=8, loc="upper left")
axes[2].set_title("Cross-Pool Mean Correlation (Rise = Panic Spreading to Uncorrelated Assets)", fontsize=11)

# Number of Components Above MP
axes[3].bar(t, df_roll["n_above_mp"], color="#aa44ff", alpha=0.7, width=0.03)
add_events(axes[3])
axes[3].set_ylabel("Components Above MP")
axes[3].set_xlabel("Time (UTC)")
axes[3].set_title("Number of Components Exceeding M-P Bound (>0 = Non-Noise Systemic Signal)", fontsize=11)

for ax in axes:
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle(
    "Aave V3 Cross-Pool Withdrawal RMT Rolling Analysis · 12H Window · Kelp DAO Bridge Exploit\n"
    "Real On-Chain Data · 29 Assets · Attack Leads Public Information by 60 Minutes",
    fontsize=13, fontweight="bold", y=1.01
)

plt.savefig("output/rmt_rolling_timeseries_e.png", dpi=150, bbox_inches="tight")
print("✓ Figure 2 saved: output/rmt_rolling_timeseries_e.png")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════
# Step 2-rmt_three_period_fixed
# ══════════════════════════════════════════════════════
import pandas as pd
import numpy as np
from scipy.linalg import eigvalsh
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Load withdrawal matrix
matrix = pd.read_csv("data/withdrawal_matrix_1h.csv", index_col=0, parse_dates=True)
matrix.index = pd.to_datetime(matrix.index, utc=True)
matrix = matrix.astype(float)

def get_spectrum(df):
    T, N = df.shape
    std = df.std()
    std[std < 1e-10] = 1e-10
    normed = (df - df.mean()) / std
    corr = (normed.T @ normed / T).values
    corr = np.nan_to_num(corr)
    
    evals = np.sort(eigvalsh(corr))[::-1]
    gamma = N / T
    lplus = (1 + np.sqrt(gamma))**2
    
    p = evals / evals.sum()
    eff_rank = np.exp(-np.sum(p * np.log(p + 1e-12)))
    mean_corr = (corr.sum() - N) / (N * (N - 1))
    
    return {
        "evals": evals, 
        "gamma": gamma, 
        "lplus": lplus,
        "corr": corr, 
        "eff_rank": eff_rank, 
        "mean_corr": mean_corr,
        "T": T, 
        "N": N
    }


# ── Fixed Window Analysis: Acute Phase uses 48H rolling window centered at 20:00 ──
# T=48 > N=29, statistically robust

ATTACK_IDX = 20
t_peak = pd.Timestamp("2026-04-18T20:00:00", tz="UTC")
peak_pos = matrix.index.get_loc(t_peak)
print(f"Peak position: {peak_pos}, Time: {matrix.index[peak_pos]}")

WINDOW = 48

t_pre  = pd.Timestamp("2026-04-18T17:00:00", tz="UTC")   # Pre-attack
t_acute = pd.Timestamp("2026-04-18T20:00:00", tz="UTC")  # Acute phase
t_post = pd.Timestamp("2026-04-19T12:00:00", tz="UTC")   # Post-panic

def get_window_spectrum(center_time, window=48):
    pos = matrix.index.get_loc(center_time)
    if pos < window:
        w = matrix.iloc[:pos]
    else:
        w = matrix.iloc[pos - window:pos]
    r = get_spectrum(w)
    r["center"] = center_time
    return r


periods_fixed = {
    "Pre-Attack\n(48H ending 17:00 Apr 18)": get_window_spectrum(t_pre),
    "Acute Phase\n(48H ending 20:00 Apr 18)": get_window_spectrum(t_acute),
    "Post-Panic\n(48H ending 12:00 Apr 19)": get_window_spectrum(t_post),
}

for name, r in periods_fixed.items():
    print(f"\n{name.split(chr(10))[0]}")
    print(f" T={r['T']}, N={r['N']}, γ={r['gamma']:.3f}, λ+={r['lplus']:.3f}")
    print(f" λ₁={r['evals'][0]:.4f}, Components above MP={np.sum(r['evals'] > r['lplus'])}")
    print(f" Effective Rank={r['eff_rank']:.2f}, Mean Corr={r['mean_corr']:.4f}")


# ── Marchenko-Pastur PDF ──────────────────────────────
def mp_pdf(x, gamma):
    lp = (1 + np.sqrt(gamma))**2
    lm = (1 - np.sqrt(gamma))**2
    return np.where(
        (x >= lm) & (x <= lp),
        np.sqrt(np.maximum((lp - x) * (x - lm), 0)) / (2 * np.pi * gamma * x),
        0.0
    )


# ── Figure: Three-Period Comparison (Fixed Window) ─────────────────────────
fig = plt.figure(figsize=(18, 10))
gs = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.32)

colors = ["#2196F3", "#FF3D00", "#FF9800"]
assets = matrix.columns.tolist()

for col, (name, res) in enumerate(periods_fixed.items()):
    evals = res["evals"]
    gamma = res["gamma"]
    lplus = res["lplus"]
    color = colors[col]
    n_above = int(np.sum(evals > lplus))

    # Top row: Eigenvalue spectrum
    ax1 = fig.add_subplot(gs[0, col])
    ax1.hist(evals, bins=18, density=True, alpha=0.65, color=color, label="Observed Eigenvalues")
    x_mp = np.linspace(0.01, lplus * 0.98, 600)
    ax1.plot(x_mp, mp_pdf(x_mp, gamma), 'k--', lw=2, label="M-P Distribution (Noise)")
    ax1.axvline(lplus, color='royalblue', ls=':', lw=2, label=f"λ+={lplus:.2f}")

    # Mark spikes above MP
    spikes = sorted(evals[evals > lplus], reverse=True)
    for sp in spikes[:3]:
        ax1.axvline(sp, color='red', lw=2, alpha=0.85)
        ax1.text(sp + 0.05, ax1.get_ylim()[1] * 0.85, f"{sp:.2f}", 
                 color='red', fontsize=8, rotation=90, va='top')

    ax1.set_title(
        f"{name}\n"
        f"λ₁={evals[0]:.3f} | Info Components={n_above} | Eff.Rank={res['eff_rank']:.1f}",
        fontsize=10
    )
    ax1.set_xlabel("Eigenvalue")
    ax1.set_ylabel("Density")
    ax1.legend(fontsize=7)

    # Bottom row: Correlation matrix heatmap
    ax2 = fig.add_subplot(gs[1, col])
    corr_df = pd.DataFrame(res["corr"], index=assets, columns=assets)
    
    # Sort assets by mean correlation for clearer structure
    row_means = corr_df.mean(axis=1).sort_values(ascending=False)
    order = row_means.index.tolist()
    corr_sorted = corr_df.loc[order, order]
   
    sns.heatmap(
        corr_sorted, ax=ax2, mask=np.eye(len(assets), dtype=bool),
        cmap="RdBu_r", center=0, vmin=-0.5, vmax=0.8,
        xticklabels=False, yticklabels=False,
        cbar_kws={"shrink": 0.7, "label": "Correlation"}
    )
    ax2.set_title(
        f"Correlation Matrix (Assets sorted by correlation)\n"
        f"mean_corr={res['mean_corr']:.4f}",
        fontsize=10
    )

# Key comparison annotation
fig.text(0.5, 0.01,
    "Pre→Acute: λ₁ ×{:.1f} | eff_rank {:.1f}→{:.1f} | mean_corr ×{:.0f}".format(
        periods_fixed["Acute Phase\n(48H ending 20:00 Apr 18)"]["evals"][0] /
        periods_fixed["Pre-Attack\n(48H ending 17:00 Apr 18)"]["evals"][0],
        periods_fixed["Pre-Attack\n(48H ending 17:00 Apr 18)"]["eff_rank"],
        periods_fixed["Acute Phase\n(48H ending 20:00 Apr 18)"]["eff_rank"],
        periods_fixed["Acute Phase\n(48H ending 20:00 Apr 18)"]["mean_corr"] /
        max(periods_fixed["Pre-Attack\n(48H ending 17:00 Apr 18)"]["mean_corr"], 0.001)
    ),
    ha='center', fontsize=11, fontweight='bold', color='darkred'
)

plt.suptitle(
    "RMT Three-Period Analysis (Fixed Window) · Aave V3 Ethereum · Kelp DAO Bridge Exploit · Apr 18–19 2026\n"
    "Real On-Chain Data · 29 Assets · 24,050 Withdrawals · Unified 48H Window (T=48 > N=29)",
    fontsize=12, fontweight="bold"
)

plt.savefig("output/rmt_three_period_fixed_e.png", dpi=150, bbox_inches="tight")
print("\n✓ Figure saved: output/rmt_three_period_fixed_e.png")
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════
# Visualization: Step 3 - Comprehensive 4-Panel Figure
# ══════════════════════════════════════════════════════

fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.30)

# ── Panel A: Cumulative Exit Curve ─────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(t_hours, cum * 100, color='#E53935', lw=2.8, label='Cumulative Exit Rate (Two-Phase Model)')
ax1.fill_between(t_hours, cum * 100, alpha=0.18, color='#E53935')
ax1.axvline(0, color='#FF9800', ls='--', lw=1.8, label='17:35 Attack')
ax1.axvline(1.4, color='#FFD700', ls=':', lw=1.8, label='19:00 News Release')
ax1.axhline(EXIT_RATE*100, color='gray', ls=':', lw=1.2, label=f'Actual Exit Rate {EXIT_RATE:.1%}')

ax1.annotate(f'Early Perceivers\n({ALPHA_EARLY:.0%})',
             xy=(1.8, ALPHA_EARLY*100 + 2), xytext=(6, 65),
             arrowprops=dict(arrowstyle='->', color='gray'), fontsize=10, color='#E53935')
ax1.annotate(f'Follow-on Exiters\n({ALPHA_FOLLOW:.0%})',
             xy=(15, 78), xytext=(22, 72),
             arrowprops=dict(arrowstyle='->', color='gray'), fontsize=10, color='#c0392b')

ax1.set_xlabel('Hours Since Attack')
ax1.set_ylabel('Cumulative Exit Rate (%)')
ax1.set_title('Umbrella Stakers Cumulative Exit Curve\nTwo-Phase Model: Early Perceivers + Followers', 
              fontsize=11, pad=15)
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)
ax1.set_xlim(0, 48)
ax1.set_ylim(0, 100)

# ── Panel B: Exit Behavior Autocorrelation Structure ──────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
lags = np.arange(1, 25)
np.random.seed(42)

# Independent decision simulation
ind_exits = np.random.exponential(3, TOTAL_STAKERS)
ind_hourly = np.array([np.sum((ind_exits >= h) & (ind_exits < h+1)) for h in range(48)])
ind_ac = [np.corrcoef(ind_hourly[:-l], ind_hourly[l:])[0,1] 
          if len(ind_hourly[:-l]) > 1 else 0 for l in lags]

# Coordinated behavior simulation
coord_hourly = np.array([exit_rate_flow[int(h/48*999)] * TOTAL_STAKERS for h in range(48)])
coord_ac = [np.corrcoef(coord_hourly[:-l], coord_hourly[l:])[0,1] 
            if len(coord_hourly[:-l]) > 1 else 0 for l in lags]

ax2.bar(lags - 0.2, ind_ac, width=0.35, color='#2196F3', alpha=0.75, label='Independent Decision (Simulated)')
ax2.bar(lags + 0.2, coord_ac, width=0.35, color='#E53935', alpha=0.75, label='Coordinated Behavior (Simulated)')
ax2.axhline(0.3, color='red', ls='--', lw=1.2, label='Significance Threshold 0.3')
ax2.axhline(0, color='black', lw=0.6)

ax2.set_xlabel('Lag (Hours)')
ax2.set_ylabel('Autocorrelation Coefficient')
ax2.set_title('Exit Behavior Autocorrelation Structure\nHigh Autocorrelation = Herding Effect = Coordinated Game', 
              fontsize=11, pad=15)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

# ── Panel C: Cooling Period Game Equilibrium ─────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(cooldown_days, p_slash_caught*100, color='#1565C0', lw=2.5, label='Probability of Slash Being Caught (%)')
ax3.plot(cooldown_days, exit_prob*100, color='#E53935', lw=2.5, label='Staker Exit Probability (%)')
ax3.plot(cooldown_days, nash_obj*100, color='#2E7D32', lw=2.8, ls='--', label='Protocol Protection Rate')
ax3.axvline(20, color='gray', ls=':', lw=1.5, label='Current Cooldown = 20 days')
ax3.axvline(nash_cd, color='#2E7D32', ls='--', lw=1.5, label=f'Nash Optimum ≈ {nash_cd:.0f} days')

ax3.set_xlabel('Cooldown Period (Days)')
ax3.set_ylabel('Probability (%)')
ax3.set_title(f'Cooldown Parameter Game Equilibrium Analysis\nNash Optimum ≈ {nash_cd:.0f} days (Current = 20 days)', 
              fontsize=11, pad=15)
ax3.legend(fontsize=9)
ax3.grid(alpha=0.3)

# ── Panel D: Exit Behavior Decomposition ───────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
categories = ['Early Perceivers\n(On-chain Visible)', 
              'News Followers\n(Social Media)', 
              'Non-Exiters\n(Rational / Unaware)']
values = [ALPHA_EARLY*100, ALPHA_FOLLOW*100, (1-EXIT_RATE)*100]
bar_colors = ['#E53935', '#FF9800', '#2196F3']

bars = ax4.bar(categories, values, color=bar_colors, alpha=0.85, width=0.55)
for bar, val in zip(bars, values):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.2,
             f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')

ax4.set_ylabel('Percentage of Total Stakers (%)')
ax4.set_title(f'Umbrella Exit Behavior Decomposition\nTotal Exit Rate = {EXIT_RATE:.1%} ({EXITED_BEFORE:,}/{TOTAL_STAKERS:,} stakers)',
              fontsize=11, pad=15)
ax4.set_ylim(0, 55)
ax4.grid(axis='y', alpha=0.3)

# ── Overall Title ───────────────────────────────────────────
plt.suptitle(
    "Umbrella Safety Module Game Theory Analysis · Kelp DAO Bridge Exploit\n"
    "80.5% of Stakers Exited Before Slash Triggered · Quantified Cooldown Design Flaw",
    fontsize=14, fontweight='bold', y=0.96
)

plt.savefig("output/umbrella_game_theory_4panel_e.png", dpi=160, bbox_inches="tight")
print("✓ Saved: output/umbrella_game_theory_4panel_e.png")
plt.show()